In [2]:
# ===== 셀 1: 라이브러리 / 경로 설정 =====
import pandas as pd
import numpy as np
import os, glob, re
import unicodedata

DATA_DIR = '/content/'  # 코랩에 업로드한 8개 csv가 있는 경로

def find_files(data_dir, keyword, exts=('.csv',)):
    keyword_nfc = unicodedata.normalize('NFC', keyword)
    matched = []
    for fname in os.listdir(data_dir):
        fname_nfc = unicodedata.normalize('NFC', fname)
        if fname_nfc.lower().endswith(exts) and keyword_nfc in fname_nfc:
            matched.append(os.path.join(data_dir, fname))
    return sorted(matched)

cost_files = find_files(DATA_DIR, '진료비')
incidence_files = find_files(DATA_DIR, '발생률')
survival_files = find_files(DATA_DIR, '생존율')
prevalence_files = find_files(DATA_DIR, '유병률')

print('진료비 파일(5개 기대):', cost_files)
print('발생률 파일(1개 기대):', incidence_files)
print('생존율 파일(1개 기대):', survival_files)
print('유병률 파일(1개 기대):', prevalence_files)

진료비 파일(5개 기대): ['/content/건강보험심사평가원_암 상병 진료비 통계_20211231.csv', '/content/건강보험심사평가원_암 상병 진료비 통계_20221231.csv', '/content/건강보험심사평가원_암 상병 진료비 통계_20231231.csv', '/content/건강보험심사평가원_암 상병 진료비 통계_2024.csv', '/content/건강보험심사평가원_암 상병 진료비 통계_2025.csv']
발생률 파일(1개 기대): ['/content/국립암센터_24개종 암발생률_20260120.csv']
생존율 파일(1개 기대): ['/content/국립암센터_24개종 암 상대생존율_20260120.csv']
유병률 파일(1개 기대): ['/content/국립암센터_24개종 암 유병률_20260120.csv']


In [3]:
# ===== 셀 2: 진료비 통계 5개년 로드 (CP949) =====
def load_cost_file(path):
    df = pd.read_csv(path, encoding='cp949')
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
    num_cols = ['환자수', '명세서청구건수', '입내원일수',
                '보험자부담금(선별포함)', '요양급여비용총액(선별포함)']
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['진료년도'] = pd.to_numeric(df['진료년도'], errors='coerce').astype('Int64')
    return df

cost_list = [load_cost_file(f) for f in cost_files]
df_cost = pd.concat(cost_list, ignore_index=True)

print(df_cost.shape)
print(sorted(df_cost['진료년도'].unique()))
df_cost.head()

(140, 7)
[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,진료년도,암 상병명,환자수,명세서청구건수,입내원일수,보험자부담금(선별포함),요양급여비용총액(선별포함)
0,2021,간암,76135,806165,1566989,545883351520,603727389660
1,2021,갑상샘(선)암,389000,1467756,1788430,249541807020,313793919350
2,2021,기타암,439376,2217477,3576756,979850240690,1105312994620
3,2021,난소암,22537,278794,595246,152030221200,170578770910
4,2021,뇌암,11624,159934,416207,124822311620,137870022590


In [4]:
# ===== 셀 3: 국립암센터 3종 로드 (UTF-8) =====
def clean_icd_code(val):
    # "00. C00-C96" -> "C00-C96"
    return re.sub(r'^\d+\.\s*', '', str(val)).strip()

def load_ncc_file(path):
    df = pd.read_csv(path, encoding='utf-8')
    df.columns = [c.strip() for c in df.columns]
    if '국제질병분류' in df.columns:
        df['국제질병분류'] = df['국제질병분류'].apply(clean_icd_code)
    return df

df_incidence = load_ncc_file(incidence_files[0])
df_survival = load_ncc_file(survival_files[0])
df_prevalence = load_ncc_file(prevalence_files[0])

print('발생률:', df_incidence.shape, '| 암종 수:', df_incidence['암종'].nunique())
print('생존율:', df_survival.shape, '| 암종 수:', df_survival['암종'].nunique())
print('유병률:', df_prevalence.shape, '| 암종 수:', df_prevalence['암종'].nunique())

발생률: (1875, 7) | 암종 수: 25
생존율: (525, 6) | 암종 수: 25
유병률: (75, 7) | 암종 수: 25


In [5]:
# ===== 셀 4: 데이터 확인 (매핑표 작성 전 사전 점검) =====
# 1) 진료비 통계 - 연도별 암 상병명 목록이 동일한지 확인
cost_by_year = df_cost.groupby('진료년도')['암 상병명'].apply(lambda s: set(s.unique()))
all_same = all(v == cost_by_year.iloc[0] for v in cost_by_year)
print('연도별 암 상병명 목록이 모두 동일한가?', all_same)
if not all_same:
    for y, s in cost_by_year.items():
        print(y, sorted(s))

cost_names = sorted(cost_by_year.iloc[0])
print(f'\n[심평원] 암 상병명 {len(cost_names)}개:')
print(cost_names)

# 2) 국립암센터 3개 파일이 같은 분류체계를 쓰는지 확인
inc_set = set(zip(df_incidence['국제질병분류'], df_incidence['암종']))
surv_set = set(zip(df_survival['국제질병분류'], df_survival['암종']))
prev_set = set(zip(df_prevalence['국제질병분류'], df_prevalence['암종']))

print('\n발생률 vs 생존율 분류체계 동일?', inc_set == surv_set)
print('발생률 vs 유병률 분류체계 동일?', inc_set == prev_set)

ncc_names = sorted(inc_set, key=lambda x: x[0])
print(f'\n[국립암센터] 암종 {len(ncc_names)}개 (코드, 이름):')
for code, name in ncc_names:
    print(code, '-', name)

# 3) 매핑표 작성용 참고 출력
print('\n--- 매핑표 작성용 참고 ---')
print('심평원 28개:', cost_names)
print('국립암센터 목록:', [n for _, n in ncc_names])

연도별 암 상병명 목록이 모두 동일한가? True

[심평원] 암 상병명 28개:
['간암', '갑상샘(선)암', '기타암', '난소암', '뇌암', '다발성골수종', '담낭암', '담도암', '대장암', '방광암', '백혈병', '비호지킨림프종', '뼈 및 관절연골암', '소장암', '식도암', '신우암', '신장암', '요관암', '위암', '유방암', '입술구강암', '자궁경부암', '전립선암', '중피성연조직암', '췌장암', '폐암', '피부암', '후두암']

발생률 vs 생존율 분류체계 동일? False
발생률 vs 유병률 분류체계 동일? False

[국립암센터] 암종 25개 (코드, 이름):
C00-C14 - 입술, 구강 및 인두
C00-C96 - 모든암
C15 - 식도
C16 - 위
C18-C20 - 대장
C22 - 간
C23-C24 - 담낭 및 기타담도
C25 - 췌장
C32 - 후두
C33-C34 - 폐
C50 - 유방
C53 - 자궁경부
C54 - 자궁체부
C56 - 난소
C61 - 전립선
C62 - 고환
C64 - 신장
C67 - 방광
C70-C72 - 뇌 및 중추신경계
C73 - 갑상선
C81 - 호지킨림프종
C82-C86,C96 - 비호지킨림프종
C90 - 다발성 골수종
C91-C95 - 백혈병
Re.C00-C96 - 기타 암

--- 매핑표 작성용 참고 ---
심평원 28개: ['간암', '갑상샘(선)암', '기타암', '난소암', '뇌암', '다발성골수종', '담낭암', '담도암', '대장암', '방광암', '백혈병', '비호지킨림프종', '뼈 및 관절연골암', '소장암', '식도암', '신우암', '신장암', '요관암', '위암', '유방암', '입술구강암', '자궁경부암', '전립선암', '중피성연조직암', '췌장암', '폐암', '피부암', '후두암']
국립암센터 목록: ['입술, 구강 및 인두', '모든암', '식도', '위', '대장', '간', '담낭 및 기타담도', '췌장', '후두', '폐', '유방', '자궁경부

In [6]:
# ===== 셀 5: 암종 매핑표 작성 =====
# 심평원 '암 상병명' -> 국립암센터 '암종' 매핑표
# - 1:1로 바로 맞는 건 그대로 매핑
# - 담낭암/담도암은 국립암센터가 'C23-C24 담낭 및 기타담도' 하나로 묶어놔서 '담낭·담도'로 통합
# - 국립암센터 25개 목록에 직접 대응이 없는 6개(뼈 및 관절연골암, 소장암, 신우암, 요관암,
#   중피성연조직암, 피부암)는 국립암센터의 잔여 카테고리인 '기타 암'(Re.C00-C96)으로 묶음
#   -> 심평원 자체 '기타암'까지 포함하면 총 7개 항목이 '기타 암' 하나로 합쳐짐
cancer_map = {
    '간암': '간',
    '갑상샘(선)암': '갑상선',
    '기타암': '기타 암',
    '난소암': '난소',
    '뇌암': '뇌 및 중추신경계',
    '다발성골수종': '다발성 골수종',
    '담낭암': '담낭·담도',
    '담도암': '담낭·담도',
    '대장암': '대장',
    '방광암': '방광',
    '백혈병': '백혈병',
    '비호지킨림프종': '비호지킨림프종',
    '뼈 및 관절연골암': '기타 암',
    '소장암': '기타 암',
    '식도암': '식도',
    '신우암': '기타 암',
    '신장암': '신장',
    '요관암': '기타 암',
    '위암': '위',
    '유방암': '유방',
    '입술구강암': '입술, 구강 및 인두',
    '자궁경부암': '자궁경부',
    '전립선암': '전립선',
    '중피성연조직암': '기타 암',
    '췌장암': '췌장',
    '폐암': '폐',
    '피부암': '기타 암',
    '후두암': '후두',
}

# 국립암센터 쪽 '담낭 및 기타담도' -> '담낭·담도'로 이름 통일 (세 파일 모두 적용)
ncc_rename = {'담낭 및 기타담도': '담낭·담도'}
for df in [df_incidence, df_survival, df_prevalence]:
    df['암종'] = df['암종'].replace(ncc_rename)

# 점검: 매핑 안 된 항목이 없는지 확인
unmapped = [k for k, v in cancer_map.items() if v is None]
print(f'매핑 안 된 심평원 암종: {len(unmapped)}개', unmapped)

# 점검: 심평원에 대응 항목 없는 국립암센터 암종 확인 ('모든암'은 합계행이라 제외)
mapped_ncc_values = set(cancer_map.values())
ncc_only = set(df_incidence['암종'].unique()) - mapped_ncc_values - {'모든암'}
print(f'심평원에 대응 데이터 없는 국립암센터 암종: {len(ncc_only)}개', ncc_only)
print('-> 이 암종들은 발생률/생존율/유병률엔 있지만 진료비 쪽엔 해당 표준암종 행이 없는 게 정상입니다.')

매핑 안 된 심평원 암종: 0개 []
심평원에 대응 데이터 없는 국립암센터 암종: 3개 {'고환', '자궁체부', '호지킨림프종'}
-> 이 암종들은 발생률/생존율/유병률엔 있지만 진료비 쪽엔 해당 표준암종 행이 없는 게 정상입니다.


In [7]:
# ===== 셀 6: 매핑 적용 (진료비 데이터에 '표준암종' 컬럼 추가 + 재집계) =====
df_cost['표준암종'] = df_cost['암 상병명'].map(cancer_map)

# 담낭암+담도암, 기타암+6개 항목처럼 여러 심평원 항목이 하나로 합쳐지는 경우,
# 비용 관련 수치는 '진료년도 + 표준암종' 기준으로 다시 합산
agg_cols = ['환자수', '명세서청구건수', '입내원일수', '보험자부담금(선별포함)', '요양급여비용총액(선별포함)']
df_cost_std = (
    df_cost
    .groupby(['진료년도', '표준암종'], as_index=False)[agg_cols]
    .sum()
)

print('원본 행 수:', len(df_cost), '-> 표준암종 집계 후 행 수:', len(df_cost_std))
print('연도당 표준암종 종류 수:', df_cost_std.groupby('진료년도')['표준암종'].nunique().unique())
df_cost_std.head(10)

원본 행 수: 140 -> 표준암종 집계 후 행 수: 105
연도당 표준암종 종류 수: [21]


,진료년도,표준암종,환자수,명세서청구건수,입내원일수,보험자부담금(선별포함),요양급여비용총액(선별포함)
0,2021,간,76135,806165,1566989,545883351520,603727389660
1,2021,갑상선,389000,1467756,1788430,249541807020,313793919350
2,2021,기타 암,495849,2707035,4489453,1221633458370,1373497170700
3,2021,난소,22537,278794,595246,152030221200,170578770910
4,2021,뇌 및 중추신경계,11624,159934,416207,124822311620,137870022590
5,2021,다발성 골수종,9669,198154,372052,217815155200,233084213720
6,2021,담낭·담도,24955,282438,704000,226979942140,251024723640
7,2021,대장,148384,1420651,2736113,774327857010,861572935950
8,2021,방광,40104,338846,556231,140704234940,157030838540
9,2021,백혈병,24273,283677,635568,365185819570,397319701430


In [8]:
# ===== 셀 7: 데이터 검증 =====
# 1) 결측치 / 중복행 점검
for name, df in [('진료비', df_cost_std), ('발생률', df_incidence),
                  ('생존율', df_survival), ('유병률', df_prevalence)]:
    na_cols = df.isna().sum()
    na_cols = na_cols[na_cols > 0]
    print(f'[{name}] 결측치 있는 컬럼:\n{na_cols if len(na_cols) else "없음"}')
    print(f'[{name}] 중복행: {df.duplicated().sum()}건\n')

# 2) 진료비 - 이상값 점검 (Track1 때와 동일한 패턴)
def value_sanity_check(df, total_col, insurer_col):
    non_positive = df[df[total_col] <= 0]
    over = df[df[insurer_col] > df[total_col]]
    print(f'{total_col} <= 0 인 행: {len(non_positive)}건')
    print(f'{insurer_col} > {total_col} 인 행 (보험자부담금이 총액 초과): {len(over)}건')
    return non_positive, over

_ = value_sanity_check(df_cost_std, '요양급여비용총액(선별포함)', '보험자부담금(선별포함)')

zero_patients_cost = df_cost_std[
    (df_cost_std['환자수'] == 0) & (df_cost_std['요양급여비용총액(선별포함)'] > 0)
]
print('환자수 0인데 비용 있는 행:', len(zero_patients_cost), '건')

# 3) 발생률 - '모든암'(합계행)과 개별 암종(기타 암 포함) 합이 일치하는지 교차검증
inc_sum_by_year = (
    df_incidence[(df_incidence['성별'] == '남녀전체') & (df_incidence['암종'] != '모든암')]
    .groupby('발생연도')['발생자수'].sum()
)
inc_total_by_year = (
    df_incidence[(df_incidence['성별'] == '남녀전체') & (df_incidence['암종'] == '모든암')]
    .set_index('발생연도')['발생자수']
)
diff = (inc_sum_by_year - inc_total_by_year).dropna()
print('\n연도별 (개별암종 합 - 모든암 합계행) 차이:')
print(diff[diff != 0] if (diff != 0).any() else '전부 일치')

[진료비] 결측치 있는 컬럼:
없음
[진료비] 중복행: 0건

[발생률] 결측치 있는 컬럼:
조발생률        125
연령표준화발생률    125
dtype: int64
[발생률] 중복행: 0건

[생존율] 결측치 있는 컬럼:
5년상대생존율    35
dtype: int64
[생존율] 중복행: 0건

[유병률] 결측치 있는 컬럼:
유병자수        5
조유병률        5
연령표준화유병률    5
dtype: int64
[유병률] 중복행: 0건

요양급여비용총액(선별포함) <= 0 인 행: 0건
보험자부담금(선별포함) > 요양급여비용총액(선별포함) 인 행 (보험자부담금이 총액 초과): 0건
환자수 0인데 비용 있는 행: 0 건

연도별 (개별암종 합 - 모든암 합계행) 차이:
전부 일치


In [9]:
# ===== 셀 8: 최종 산출 (4개 파일로 분리 저장) =====
CLEAN_DIR = os.path.join(DATA_DIR, 'cleaned')
os.makedirs(CLEAN_DIR, exist_ok=True)

df_cost_std.to_csv(os.path.join(CLEAN_DIR, 'cancer_cost.csv'), index=False, encoding='utf-8-sig')
df_incidence.to_csv(os.path.join(CLEAN_DIR, 'cancer_incidence.csv'), index=False, encoding='utf-8-sig')
df_survival.to_csv(os.path.join(CLEAN_DIR, 'cancer_survival.csv'), index=False, encoding='utf-8-sig')
df_prevalence.to_csv(os.path.join(CLEAN_DIR, 'cancer_prevalence.csv'), index=False, encoding='utf-8-sig')

print('저장 완료:', os.listdir(CLEAN_DIR))

저장 완료: ['cancer_incidence.csv', 'cancer_survival.csv', 'cancer_cost.csv', 'cancer_prevalence.csv']
